# 📈 AI Sales Prediction & Advertising Analytics
## Exploratory Data Analysis & Model Evaluation Notebook

This notebook demonstrates the end-to-end Machine Learning pipeline for predicting product sales based on multi-channel advertising expenditures (TV, Radio, Newspaper).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Load Dataset & Data Cleaning

In [ ]:
df = pd.read_csv('../advertising.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print("Dataset Shape:", df.shape)
print("Missing Values:\n", df.isnull().sum())
df.head()

### 2. Exploratory Data Analysis & Visualizations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.regplot(x='TV', y='Sales', data=df, ax=axes[0], color='blue')
axes[0].set_title('TV vs Sales')
sns.regplot(x='Radio', y='Sales', data=df, ax=axes[1], color='green')
axes[1].set_title('Radio vs Sales')
sns.regplot(x='Newspaper', y='Sales', data=df, ax=axes[2], color='orange')
axes[2].set_title('Newspaper vs Sales')
plt.tight_layout()
plt.show()

### 3. Feature Engineering

In [ ]:
df['Total_Spend'] = df['TV'] + df['Radio'] + df['Newspaper']
df['TV_x_Radio'] = df['TV'] * df['Radio']
df['Log_TV'] = np.log1p(df['TV'])
df['Log_Radio'] = np.log1p(df['Radio'])
df.head()

### 4. Model Training & Comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df.drop(columns=['Sales'])
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results.append({'Model': name, 'Test R2': round(r2, 4), 'RMSE': round(rmse, 4)})

pd.DataFrame(results).sort_values(by='Test R2', ascending=False)